# Escala
## Dynamic Programming

Atividade revisada do Desafio Motiva: dados sintéticos, buscas manuais, percurso em linha e planejamento de turnos com filas e pilhas de listas.

Execute as células de cima para baixo, com todos os arquivos `.py` na mesma pasta deste notebook. A execução recria os CSVs. Os algoritmos ficam nos módulos do repositório; os experimentos e suas saídas aparecem neste notebook.

Python 3.10 ou superior. A solução usa a biblioteca padrão; Jupyter ou outra ferramenta compatível é necessário somente para abrir e executar o notebook.

## Etapa 0 — Dataset sintético

Primeiro criamos a linha, necessária para associar as escadas às estações. Em seguida geramos 30 escadas com 240 leituras horárias cada, usando a semente 42. Os perfis são 18 saudáveis, 9 em desgaste gradual e 3 com falhas pontuais. Grupo de criticidade e perfil de saúde são classificações distintas.

Os blocos de escadas estão embaralhados no CSV e as leituras de cada equipamento estão em ordem temporal. Os sensores e os tempos são simulados para a atividade acadêmica.

In [1]:
from percurso import criar_linha
from gerar_dataset import gerar_dataset, gerar_resumos

criar_linha()
leituras, cadastros = gerar_dataset(30, 240, 42)
gerar_resumos(leituras, cadastros)

print("Escadas:", len(cadastros))
print("Leituras:", len(leituras))
print("Semente:", 42)


Escadas: 30
Leituras: 7200
Semente: 42


### Resumo por sensor

| sensor | minimo | media | maximo |
| --- | --- | --- | --- |
| vibracao | 1.7579 | 2.5176 | 5.0783 |
| temperatura | 42.8032 | 49.4527 | 81.8106 |
| corrente | 13.7969 | 18.1459 | 34.1689 |
| tensao | 370.7878 | 380.8636 | 418.1446 |
| nivel_oleo | 37.663 | 78.9403 | 89.444 |
| velocidade | 0.472 | 0.4991 | 0.5094 |

### Distribuição dos perfis

| perfil | quantidade |
| --- | --- |
| desgaste_gradual | 9 |
| saudavel | 18 |
| falhas_pontuais | 3 |

Os extremos incluem os valores anormais propositalmente gerados. Médias gerais podem esconder essas leituras; por isso a regra de alerta examina cada registro. Estas faixas não são limites operacionais validados pela Motiva.

## Atividade 1 — Busca por ID

A busca linear usa os 30 IDs únicos na ordem do CSV; a binária usa esses IDs ordenados. São realizadas 1.000 consultas sorteadas com reposição e semente 42. As duas buscas recebem o mesmo ID em cada consulta.

**Critério de contagem:** uma comparação representa uma posição consultada pela busca. Na binária, não contamos separadamente o teste de igualdade e o teste que decide o lado. A leitura do CSV, a remoção dos IDs repetidos e a ordenação não entram nessa contagem. Não foram medidos tempos de execução.

In [2]:
import random
import csv

from carregar_dados import escadas
from busca_linear import busca_linear
from busca_binaria import busca_binaria

escadas_ordenadas = sorted(escadas)

random.seed(42)

total_linear = 0
total_binaria = 0

for i in range(1000):
    procurado = random.choice(escadas)

    posicao_linear, comparacoes_linear = busca_linear(
        escadas, procurado
    )

    posicao_binaria, comparacoes_binaria = busca_binaria(
        escadas_ordenadas, procurado
    )

    total_linear += comparacoes_linear
    total_binaria += comparacoes_binaria

print("Quantidade de consultas:", 1000)

print("\nBusca linear:")
print("Total de comparações:", total_linear)
print("Média por consulta:", total_linear / 1000)

print("\nBusca binária:")
print("Total de comparações:", total_binaria)
print("Média por consulta:", total_binaria / 1000)

with open(
    "comparacao_buscas.csv",
    "w",
    newline="",
    encoding="utf-8"
) as arquivo:
    escritor = csv.writer(arquivo)

    escritor.writerow([
        "busca",
        "consultas",
        "total_comparacoes",
        "media_comparacoes"
    ])

    escritor.writerow([
        "linear",
        1000,
        total_linear,
        total_linear / 1000
    ])

    escritor.writerow([
        "binaria",
        1000,
        total_binaria,
        total_binaria / 1000
    ])

Quantidade de consultas: 1000

Busca linear:
Total de comparações: 15569
Média por consulta: 15.569

Busca binária:
Total de comparações: 4082
Média por consulta: 4.082


### Resultados das consultas

| busca | consultas | total_comparacoes | media_comparacoes |
| --- | --- | --- | --- |
| linear | 1000 | 15569 | 15.569 |
| binaria | 1000 | 4082 | 4.082 |

A binária consulta menos posições neste experimento, pois descarta metade do intervalo a cada repetição. A linear percorre a lista desde o começo. O experimento usa IDs existentes; verificamos os casos ausentes separadamente abaixo.

In [3]:
from carregar_dados import escadas
from busca_linear import busca_linear
from busca_binaria import busca_binaria

ordenadas = sorted(escadas)
for alvo in ["SIM-ECR-001", "SIM-ECR-999"]:
    pos_linear, cont_linear = busca_linear(escadas, alvo)
    pos_binaria, cont_binaria = busca_binaria(ordenadas, alvo)
    print(alvo, "| Linear:", pos_linear, cont_linear, "| Binária:", pos_binaria, cont_binaria)


SIM-ECR-001 | Linear: 29 30 | Binária: 0 4
SIM-ECR-999 | Linear: -1 30 | Binária: -1 5


### Primeira leitura com cinco alertas acumulados

Uma leitura conta como um alerta se ao menos um dos seis sensores sair da faixa normal, mesmo quando vários sensores estão alterados. O acumulado nunca diminui. A busca binária guarda uma posição que atende ao limite e continua à esquerda para encontrar a primeira. Quando a escada não chega a cinco alertas, ambas retornam -1 e o horário fica vazio no CSV.

O preparo dos acumulados fica fora das contagens das buscas. As posições são relativas às leituras de cada escada e começam em zero.

In [4]:
import csv

from carregar_dados import escadas
from alertas import leitura_em_alerta


def buscar_primeiro_limite_linear(acumulados, limite):
    comparacoes = 0

    for i in range(len(acumulados)):
        comparacoes += 1

        if acumulados[i] >= limite:
            return i, comparacoes

    return -1, comparacoes


def buscar_primeiro_limite_binaria(acumulados, limite):
    inicio = 0
    fim = len(acumulados) - 1
    posicao = -1
    comparacoes = 0

    while inicio <= fim:
        meio = (inicio + fim) // 2
        comparacoes += 1

        if acumulados[meio] >= limite:
            posicao = meio
            fim = meio - 1
        else:
            inicio = meio + 1

    return posicao, comparacoes


def carregar_leituras():
    leituras = []

    with open("sensores.csv", encoding="utf-8", newline="") as arquivo:
        leitor = csv.reader(arquivo)
        next(leitor)

        for leitura in leitor:
            leituras.append(leitura)

    return leituras


if __name__ == "__main__":
    leituras = carregar_leituras()
    resultados = []

    total_linear = 0
    total_binaria = 0
    escadas_com_limite = 0

    for id_escada in escadas:
        horarios = []
        acumulados = []
        total_alertas = 0

        for leitura in leituras:
            if leitura[0] == id_escada:
                if leitura_em_alerta(leitura):
                    total_alertas += 1

                horarios.append(leitura[3])
                acumulados.append(total_alertas)

        posicao_linear, comparacoes_linear = buscar_primeiro_limite_linear(
            acumulados, 5
        )

        posicao_binaria, comparacoes_binaria = buscar_primeiro_limite_binaria(
            acumulados, 5
        )

        if posicao_linear != posicao_binaria:
            raise ValueError(
                "As buscas encontraram posições diferentes: " + id_escada
            )

        if posicao_linear == -1:
            horario = ""
            situacao = "Limite não atingido"
        else:
            horario = horarios[posicao_linear]
            situacao = "Limite atingido"
            escadas_com_limite += 1

        resultados.append([
            id_escada,
            len(acumulados),
            total_alertas,
            situacao,
            posicao_linear,
            posicao_binaria,
            horario,
            comparacoes_linear,
            comparacoes_binaria
        ])

        total_linear += comparacoes_linear
        total_binaria += comparacoes_binaria

        print(
            id_escada,
            "|", situacao,
            "| Linear:", comparacoes_linear,
            "| Binária:", comparacoes_binaria
        )

    with open(
        "comparacao_alertas.csv",
        "w",
        encoding="utf-8",
        newline=""
    ) as arquivo:
        escritor = csv.writer(arquivo)

        escritor.writerow([
            "id_escada",
            "quantidade_leituras",
            "total_alertas",
            "situacao",
            "posicao_linear",
            "posicao_binaria",
            "horario_quinto_alerta",
            "comparacoes_linear",
            "comparacoes_binaria"
        ])

        escritor.writerows(resultados)

    print("\nEscadas analisadas:", len(escadas))
    print("Escadas que atingiram cinco alertas:", escadas_com_limite)
    print("Total de comparações da linear:", total_linear)
    print("Total de comparações da binária:", total_binaria)
    print("Arquivo gerado: comparacao_alertas.csv")

SIM-ECR-006 | Limite não atingido | Linear: 240 | Binária: 8
SIM-ECR-008 | Limite não atingido | Linear: 240 | Binária: 8
SIM-ECR-029 | Limite não atingido | Linear: 240 | Binária: 8
SIM-ECR-028 | Limite não atingido | Linear: 240 | Binária: 8
SIM-ECR-007 | Limite não atingido | Linear: 240 | Binária: 8
SIM-ECR-024 | Limite atingido | Linear: 101 | Binária: 8
SIM-ECR-010 | Limite não atingido | Linear: 240 | Binária: 8
SIM-ECR-023 | Limite não atingido | Linear: 240 | Binária: 8
SIM-ECR-012 | Limite atingido | Linear: 148 | Binária: 8
SIM-ECR-021 | Limite não atingido | Linear: 240 | Binária: 8
SIM-ECR-009 | Limite não atingido | Linear: 240 | Binária: 8
SIM-ECR-030 | Limite atingido | Linear: 187 | Binária: 8
SIM-ECR-016 | Limite atingido | Linear: 131 | Binária: 8
SIM-ECR-018 | Limite não atingido | Linear: 240 | Binária: 8
SIM-ECR-025 | Limite não atingido | Linear: 240 | Binária: 8
SIM-ECR-017 | Limite não atingido | Linear: 240 | Binária: 8
SIM-ECR-014 | Limite atingido | Linear: 

### Comparação para todas as escadas

| id_escada | quantidade_leituras | total_alertas | situacao | posicao_linear | posicao_binaria | horario_quinto_alerta | comparacoes_linear | comparacoes_binaria |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| SIM-ECR-006 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-008 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-029 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-028 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-007 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-024 | 240 | 142 | Limite atingido | 100 | 100 | 2026-08-05T04:00:00 | 101 | 8 |
| SIM-ECR-010 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-023 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-012 | 240 | 96 | Limite atingido | 147 | 147 | 2026-08-07T03:00:00 | 148 | 8 |
| SIM-ECR-021 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-009 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-030 | 240 | 58 | Limite atingido | 186 | 186 | 2026-08-08T18:00:00 | 187 | 8 |
| SIM-ECR-016 | 240 | 6 | Limite atingido | 130 | 130 | 2026-08-06T10:00:00 | 131 | 8 |
| SIM-ECR-018 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-025 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-017 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-014 | 240 | 149 | Limite atingido | 88 | 88 | 2026-08-04T16:00:00 | 89 | 8 |
| SIM-ECR-002 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-020 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-013 | 240 | 6 | Limite atingido | 144 | 144 | 2026-08-07T00:00:00 | 145 | 8 |
| SIM-ECR-026 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-022 | 240 | 6 | Limite atingido | 174 | 174 | 2026-08-08T06:00:00 | 175 | 8 |
| SIM-ECR-004 | 240 | 95 | Limite atingido | 148 | 148 | 2026-08-07T04:00:00 | 149 | 8 |
| SIM-ECR-011 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-019 | 240 | 57 | Limite atingido | 185 | 185 | 2026-08-08T17:00:00 | 186 | 8 |
| SIM-ECR-005 | 240 | 126 | Limite atingido | 112 | 112 | 2026-08-05T16:00:00 | 113 | 8 |
| SIM-ECR-027 | 240 | 152 | Limite atingido | 87 | 87 | 2026-08-04T15:00:00 | 88 | 8 |
| SIM-ECR-015 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-003 | 240 | 0 | Limite não atingido | -1 | -1 |  | 240 | 8 |
| SIM-ECR-001 | 240 | 140 | Limite atingido | 103 | 103 | 2026-08-05T07:00:00 | 104 | 8 |

As duas buscas devem retornar a mesma posição para cada equipamento. A binária reduz as posições consultadas nos acumulados; a preparação e o agrupamento das leituras continuam necessários. As escadas saudáveis não atingem o limite neste dataset.

## Atividade 2 — Percurso com pilha

Usamos oito estações consecutivas da Linha 4-Amarela, de Luz até Pinheiros. A base de manutenção é representada por Luz. Os nomes e a ordem seguem a [página da Motiva](https://trilhos.motiva.com.br/viaquatro/linha-4---amarela/); os minutos entre estações são hipóteses da simulação.

`tempo_ate` encontra a estação com busca linear e soma os trechos desde a base. Na ida, `rota_ida_e_volta` usa `append` para empilhar as estações; na volta usa `pop` para recuperar a ordem inversa. Assumimos o mesmo tempo nos dois sentidos e não incluímos espera ou transferência.

In [5]:
import csv


def criar_linha():
    # Estação e minutos até a próxima.
    # Tempos simulados para a atividade.
    linha = [
        ["Luz", 3],
        ["República", 3],
        ["Higienópolis-Mackenzie", 2],
        ["Paulista", 3],
        ["Oscar Freire", 3],
        ["Fradique Coutinho", 2],
        ["Faria Lima", 3],
        ["Pinheiros", 0]
    ]

    with open(
        "linha.csv", "w", encoding="utf-8", newline=""
    ) as arquivo:
        escritor = csv.writer(arquivo)
        escritor.writerow(["estacao", "minutos_proxima"])
        escritor.writerows(linha)


def carregar_linha():
    linha = []

    with open(
        "linha.csv", encoding="utf-8", newline=""
    ) as arquivo:
        leitor = csv.reader(arquivo)
        next(leitor)

        for registro in leitor:
            estacao = registro[0]
            minutos = int(registro[1])

            linha.append([estacao, minutos])

    return linha


def tempo_ate(linha, estacao):
    """Busca a estação e soma os minutos desde a base."""
    minutos = 0

    for i in range(len(linha)):
        if linha[i][0] == estacao:
            return minutos

        minutos += linha[i][1]

    raise ValueError("Estação não encontrada: " + estacao)


def rota_ida_e_volta(linha, estacao):
    """Empilha as estações na ida e desempilha na volta."""
    ida = []
    volta = []
    pilha = []
    encontrada = False

    for i in range(len(linha)):
        nome = linha[i][0]

        ida.append(nome)
        pilha.append(nome)

        if nome == estacao:
            encontrada = True
            break

    if encontrada == False:
        raise ValueError("Estação não encontrada: " + estacao)

    while pilha:
        nome = pilha.pop()
        volta.append(nome)

    return ida, volta


def salvar_tempos(linha):
    resultados = []

    for i in range(len(linha)):
        estacao = linha[i][0]
        minutos_ida = tempo_ate(linha, estacao)

        resultados.append([
            estacao,
            minutos_ida,
            minutos_ida * 2
        ])

    with open(
        "tempos_base.csv", "w", encoding="utf-8", newline=""
    ) as arquivo:
        escritor = csv.writer(arquivo)

        escritor.writerow([
            "estacao",
            "minutos_ida",
            "minutos_ida_volta"
        ])

        escritor.writerows(resultados)


if __name__ == "__main__":
    criar_linha()
    linha = carregar_linha()
    salvar_tempos(linha)

    print("Base:", linha[0][0])

    for i in range(len(linha)):
        estacao = linha[i][0]

        minutos = tempo_ate(linha, estacao)
        ida, volta = rota_ida_e_volta(linha, estacao)

        print("\nEstação:", estacao)
        print("Ida:", " -> ".join(ida))
        print("Volta:", " -> ".join(volta))
        print("Minutos de ida:", minutos)
        print("Minutos de ida e volta:", minutos * 2)

    print("\nArquivos gerados: linha.csv e tempos_base.csv")

Base: Luz

Estação: Luz
Ida: Luz
Volta: Luz
Minutos de ida: 0
Minutos de ida e volta: 0

Estação: República
Ida: Luz -> República
Volta: República -> Luz
Minutos de ida: 3
Minutos de ida e volta: 6

Estação: Higienópolis-Mackenzie
Ida: Luz -> República -> Higienópolis-Mackenzie
Volta: Higienópolis-Mackenzie -> República -> Luz
Minutos de ida: 6
Minutos de ida e volta: 12

Estação: Paulista
Ida: Luz -> República -> Higienópolis-Mackenzie -> Paulista
Volta: Paulista -> Higienópolis-Mackenzie -> República -> Luz
Minutos de ida: 8
Minutos de ida e volta: 16

Estação: Oscar Freire
Ida: Luz -> República -> Higienópolis-Mackenzie -> Paulista -> Oscar Freire
Volta: Oscar Freire -> Paulista -> Higienópolis-Mackenzie -> República -> Luz
Minutos de ida: 11
Minutos de ida e volta: 22

Estação: Fradique Coutinho
Ida: Luz -> República -> Higienópolis-Mackenzie -> Paulista -> Oscar Freire -> Fradique Coutinho
Volta: Fradique Coutinho -> Oscar Freire -> Paulista -> Higienópolis-Mackenzie -> República 

### Tempos desde a base

| estacao | minutos_ida | minutos_ida_volta |
| --- | --- | --- |
| Luz | 0 | 0 |
| República | 3 | 6 |
| Higienópolis-Mackenzie | 6 | 12 |
| Paulista | 8 | 16 |
| Oscar Freire | 11 | 22 |
| Fradique Coutinho | 14 | 28 |
| Faria Lima | 16 | 32 |
| Pinheiros | 19 | 38 |

Pinheiros fica a 19 minutos da base neste cenário, ou 38 minutos de ida e volta. Para a própria base, o deslocamento é zero; um serviço realizado nela ainda consome seu tempo de manutenção.

## Atividade 3 — Plano de manutenção

Cada escada entra uma única vez na fila, pelo horário da primeira leitura em alerta. Este critério é diferente do quinto alerta usado no experimento de buscas. Ordenamos cada grupo pelo primeiro alerta e desempate por ID. A fila crítica é processada antes da normal e cada retirada usa `pop(0)`.

Cada atendimento inclui ida, 60 minutos de serviço para críticas ou 40 para normais e volta à base. Os horários de início e fim estão em minutos desde o início do turno, incluindo os deslocamentos. Se um atendimento não cabe, ele vai para pendentes e os seguintes são examinados.

Na redução de uma hora, a agenda funciona como pilha: removemos os últimos atendimentos com `pop()` até caber. Não preenchemos novamente a folga. O planejamento é uma fotografia das ocorrências do dataset, sem simular chegada de novos alertas durante o turno.

In [6]:
import csv

from alertas import leitura_em_alerta
from percurso import carregar_linha, tempo_ate, rota_ida_e_volta


def carregar_primeiros_alertas():
    """Guarda somente a primeira leitura em alerta de cada escada."""
    primeiros = {}

    with open(
        "sensores.csv", encoding="utf-8", newline=""
    ) as arquivo:
        leitor = csv.reader(arquivo)
        next(leitor)

        for leitura in leitor:
            id_escada = leitura[0]

            if leitura_em_alerta(leitura):
                if id_escada not in primeiros:
                    primeiros[id_escada] = {
                        "id_escada": id_escada,
                        "estacao": leitura[1],
                        "grupo": leitura[2],
                        "primeiro_alerta": leitura[3]
                    }

    return list(primeiros.values())


def montar_filas(alertas):
    fila_critica = []
    fila_normal = []

    # Datas ISO podem ser ordenadas como texto neste dataset.
    ordenados = sorted(
        alertas,
        key=lambda escada: (
            escada["primeiro_alerta"],
            escada["id_escada"]
        )
    )

    for escada in ordenados:
        if escada["grupo"] == "crítica":
            fila_critica.append(escada)
        else:
            fila_normal.append(escada)

    return fila_critica, fila_normal


def planejar_turno(base, horas):
    linha = carregar_linha()

    if base != linha[0][0]:
        raise ValueError("A base deve ser a primeira estação da linha.")

    if horas < 0:
        raise ValueError("A duração do turno não pode ser negativa.")

    alertas = carregar_primeiros_alertas()
    fila_critica, fila_normal = montar_filas(alertas)

    agenda = []
    pendentes = []
    tempo_usado = 0
    limite = horas * 60

    # Percorre toda a fila crítica antes da normal.
    for fila in [fila_critica, fila_normal]:
        while fila:
            escada = fila.pop(0)

            if escada["grupo"] == "crítica":
                servico = 60
            else:
                servico = 40

            ida = tempo_ate(linha, escada["estacao"])
            custo = ida + servico + ida

            if tempo_usado + custo <= limite:
                rota_ida, rota_volta = rota_ida_e_volta(
                    linha, escada["estacao"]
                )

                atendimento = {
                    "id_escada": escada["id_escada"],
                    "estacao": escada["estacao"],
                    "grupo": escada["grupo"],
                    "primeiro_alerta": escada["primeiro_alerta"],
                    "rota_ida": " -> ".join(rota_ida),
                    "rota_volta": " -> ".join(rota_volta),
                    "inicio_minutos": tempo_usado,
                    "fim_minutos": tempo_usado + custo,
                    "minutos_servico": servico
                }

                agenda.append(atendimento)
                tempo_usado += custo
            else:
                pendentes.append(escada)

    return agenda, pendentes


def reduzir_turno(agenda, pendentes, novas_horas):
    """Desfaz os últimos atendimentos até a agenda caber."""
    if novas_horas < 0:
        raise ValueError("A duração do turno não pode ser negativa.")

    # Copia as listas para preservar o plano original.
    nova_agenda = list(agenda)
    novos_pendentes = list(pendentes)
    limite = novas_horas * 60

    while nova_agenda:
        if nova_agenda[-1]["fim_minutos"] <= limite:
            break

        removido = nova_agenda.pop()

        novos_pendentes.append({
            "id_escada": removido["id_escada"],
            "estacao": removido["estacao"],
            "grupo": removido["grupo"],
            "primeiro_alerta": removido["primeiro_alerta"]
        })

    return nova_agenda, novos_pendentes


def salvar_plano(nome, agenda, pendentes):
    with open(
        nome + "_agenda.csv", "w", encoding="utf-8", newline=""
    ) as arquivo:
        colunas = [
            "id_escada",
            "estacao",
            "grupo",
            "primeiro_alerta",
            "rota_ida",
            "rota_volta",
            "inicio_minutos",
            "fim_minutos",
            "minutos_servico"
        ]

        escritor = csv.DictWriter(arquivo, fieldnames=colunas)
        escritor.writeheader()
        escritor.writerows(agenda)

    with open(
        nome + "_pendentes.csv", "w", encoding="utf-8", newline=""
    ) as arquivo:
        colunas = [
            "id_escada",
            "estacao",
            "grupo",
            "primeiro_alerta"
        ]

        escritor = csv.DictWriter(arquivo, fieldnames=colunas)
        escritor.writeheader()
        escritor.writerows(pendentes)


def mostrar_resumo(nome, agenda, pendentes):
    tempo_usado = 0

    if agenda:
        tempo_usado = agenda[-1]["fim_minutos"]

    print("\nPlano:", nome)
    print("Atendimentos:", len(agenda))
    print("Pendentes:", len(pendentes))
    print("Minutos utilizados:", tempo_usado)


if __name__ == "__main__":
    linha = carregar_linha()
    base = linha[0][0]

    for horas in [4, 6, 8]:
        agenda, pendentes = planejar_turno(base, horas)

        nome = "turno_" + str(horas) + "h"
        salvar_plano(nome, agenda, pendentes)
        mostrar_resumo(nome, agenda, pendentes)

        agenda_reduzida, pendentes_reduzidos = reduzir_turno(
            agenda, pendentes, horas - 1
        )

        nome_reduzido = nome + "_menos_1h"

        salvar_plano(
            nome_reduzido,
            agenda_reduzida,
            pendentes_reduzidos
        )

        mostrar_resumo(
            nome_reduzido,
            agenda_reduzida,
            pendentes_reduzidos
        )


Plano: turno_4h
Atendimentos: 3
Pendentes: 9
Minutos utilizados: 238

Plano: turno_4h_menos_1h
Atendimentos: 2
Pendentes: 10
Minutos utilizados: 170

Plano: turno_6h
Atendimentos: 4
Pendentes: 8
Minutos utilizados: 334

Plano: turno_6h_menos_1h
Atendimentos: 3
Pendentes: 9
Minutos utilizados: 246

Plano: turno_8h
Atendimentos: 6
Pendentes: 6
Minutos utilizados: 480

Plano: turno_8h_menos_1h
Atendimentos: 5
Pendentes: 7
Minutos utilizados: 412


### Turno de 4 horas

**Agenda**

| id_escada | estacao | grupo | primeiro_alerta | rota_ida | rota_volta | inicio_minutos | fim_minutos | minutos_servico |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| SIM-ECR-027 | Higienópolis-Mackenzie | crítica | 2026-08-04T05:00:00 | Luz -> República -> Higienópolis-Mackenzie | Higienópolis-Mackenzie -> República -> Luz | 0 | 72 | 60 |
| SIM-ECR-024 | Pinheiros | crítica | 2026-08-04T18:00:00 | Luz -> República -> Higienópolis-Mackenzie -> Paulista -> Oscar Freire -> Fradique Coutinho -> Faria Lima -> Pinheiros | Pinheiros -> Faria Lima -> Fradique Coutinho -> Oscar Freire -> Paulista -> Higienópolis-Mackenzie -> República -> Luz | 72 | 170 | 60 |
| SIM-ECR-022 | Fradique Coutinho | não crítica | 2026-08-02T16:00:00 | Luz -> República -> Higienópolis-Mackenzie -> Paulista -> Oscar Freire -> Fradique Coutinho | Fradique Coutinho -> Oscar Freire -> Paulista -> Higienópolis-Mackenzie -> República -> Luz | 170 | 238 | 40 |

**Pendentes**

| id_escada | estacao | grupo | primeiro_alerta |
| --- | --- | --- | --- |
| SIM-ECR-012 | Paulista | crítica | 2026-08-06T21:00:00 |
| SIM-ECR-030 | Fradique Coutinho | crítica | 2026-08-08T10:00:00 |
| SIM-ECR-016 | Pinheiros | não crítica | 2026-08-01T03:00:00 |
| SIM-ECR-013 | Oscar Freire | não crítica | 2026-08-03T21:00:00 |
| SIM-ECR-014 | Fradique Coutinho | não crítica | 2026-08-04T04:00:00 |
| SIM-ECR-001 | Luz | não crítica | 2026-08-05T00:00:00 |
| SIM-ECR-005 | Oscar Freire | não crítica | 2026-08-05T10:00:00 |
| SIM-ECR-004 | Paulista | não crítica | 2026-08-06T20:00:00 |
| SIM-ECR-019 | Higienópolis-Mackenzie | não crítica | 2026-08-08T05:00:00 |

### Redução de 4 para 3 horas

**Agenda após a redução**

| id_escada | estacao | grupo | primeiro_alerta | rota_ida | rota_volta | inicio_minutos | fim_minutos | minutos_servico |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| SIM-ECR-027 | Higienópolis-Mackenzie | crítica | 2026-08-04T05:00:00 | Luz -> República -> Higienópolis-Mackenzie | Higienópolis-Mackenzie -> República -> Luz | 0 | 72 | 60 |
| SIM-ECR-024 | Pinheiros | crítica | 2026-08-04T18:00:00 | Luz -> República -> Higienópolis-Mackenzie -> Paulista -> Oscar Freire -> Fradique Coutinho -> Faria Lima -> Pinheiros | Pinheiros -> Faria Lima -> Fradique Coutinho -> Oscar Freire -> Paulista -> Higienópolis-Mackenzie -> República -> Luz | 72 | 170 | 60 |

**Pendentes após a redução**

| id_escada | estacao | grupo | primeiro_alerta |
| --- | --- | --- | --- |
| SIM-ECR-012 | Paulista | crítica | 2026-08-06T21:00:00 |
| SIM-ECR-030 | Fradique Coutinho | crítica | 2026-08-08T10:00:00 |
| SIM-ECR-016 | Pinheiros | não crítica | 2026-08-01T03:00:00 |
| SIM-ECR-013 | Oscar Freire | não crítica | 2026-08-03T21:00:00 |
| SIM-ECR-014 | Fradique Coutinho | não crítica | 2026-08-04T04:00:00 |
| SIM-ECR-001 | Luz | não crítica | 2026-08-05T00:00:00 |
| SIM-ECR-005 | Oscar Freire | não crítica | 2026-08-05T10:00:00 |
| SIM-ECR-004 | Paulista | não crítica | 2026-08-06T20:00:00 |
| SIM-ECR-019 | Higienópolis-Mackenzie | não crítica | 2026-08-08T05:00:00 |
| SIM-ECR-022 | Fradique Coutinho | não crítica | 2026-08-02T16:00:00 |


### Turno de 6 horas

**Agenda**

| id_escada | estacao | grupo | primeiro_alerta | rota_ida | rota_volta | inicio_minutos | fim_minutos | minutos_servico |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| SIM-ECR-027 | Higienópolis-Mackenzie | crítica | 2026-08-04T05:00:00 | Luz -> República -> Higienópolis-Mackenzie | Higienópolis-Mackenzie -> República -> Luz | 0 | 72 | 60 |
| SIM-ECR-024 | Pinheiros | crítica | 2026-08-04T18:00:00 | Luz -> República -> Higienópolis-Mackenzie -> Paulista -> Oscar Freire -> Fradique Coutinho -> Faria Lima -> Pinheiros | Pinheiros -> Faria Lima -> Fradique Coutinho -> Oscar Freire -> Paulista -> Higienópolis-Mackenzie -> República -> Luz | 72 | 170 | 60 |
| SIM-ECR-012 | Paulista | crítica | 2026-08-06T21:00:00 | Luz -> República -> Higienópolis-Mackenzie -> Paulista | Paulista -> Higienópolis-Mackenzie -> República -> Luz | 170 | 246 | 60 |
| SIM-ECR-030 | Fradique Coutinho | crítica | 2026-08-08T10:00:00 | Luz -> República -> Higienópolis-Mackenzie -> Paulista -> Oscar Freire -> Fradique Coutinho | Fradique Coutinho -> Oscar Freire -> Paulista -> Higienópolis-Mackenzie -> República -> Luz | 246 | 334 | 60 |

**Pendentes**

| id_escada | estacao | grupo | primeiro_alerta |
| --- | --- | --- | --- |
| SIM-ECR-016 | Pinheiros | não crítica | 2026-08-01T03:00:00 |
| SIM-ECR-022 | Fradique Coutinho | não crítica | 2026-08-02T16:00:00 |
| SIM-ECR-013 | Oscar Freire | não crítica | 2026-08-03T21:00:00 |
| SIM-ECR-014 | Fradique Coutinho | não crítica | 2026-08-04T04:00:00 |
| SIM-ECR-001 | Luz | não crítica | 2026-08-05T00:00:00 |
| SIM-ECR-005 | Oscar Freire | não crítica | 2026-08-05T10:00:00 |
| SIM-ECR-004 | Paulista | não crítica | 2026-08-06T20:00:00 |
| SIM-ECR-019 | Higienópolis-Mackenzie | não crítica | 2026-08-08T05:00:00 |

### Redução de 6 para 5 horas

**Agenda após a redução**

| id_escada | estacao | grupo | primeiro_alerta | rota_ida | rota_volta | inicio_minutos | fim_minutos | minutos_servico |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| SIM-ECR-027 | Higienópolis-Mackenzie | crítica | 2026-08-04T05:00:00 | Luz -> República -> Higienópolis-Mackenzie | Higienópolis-Mackenzie -> República -> Luz | 0 | 72 | 60 |
| SIM-ECR-024 | Pinheiros | crítica | 2026-08-04T18:00:00 | Luz -> República -> Higienópolis-Mackenzie -> Paulista -> Oscar Freire -> Fradique Coutinho -> Faria Lima -> Pinheiros | Pinheiros -> Faria Lima -> Fradique Coutinho -> Oscar Freire -> Paulista -> Higienópolis-Mackenzie -> República -> Luz | 72 | 170 | 60 |
| SIM-ECR-012 | Paulista | crítica | 2026-08-06T21:00:00 | Luz -> República -> Higienópolis-Mackenzie -> Paulista | Paulista -> Higienópolis-Mackenzie -> República -> Luz | 170 | 246 | 60 |

**Pendentes após a redução**

| id_escada | estacao | grupo | primeiro_alerta |
| --- | --- | --- | --- |
| SIM-ECR-016 | Pinheiros | não crítica | 2026-08-01T03:00:00 |
| SIM-ECR-022 | Fradique Coutinho | não crítica | 2026-08-02T16:00:00 |
| SIM-ECR-013 | Oscar Freire | não crítica | 2026-08-03T21:00:00 |
| SIM-ECR-014 | Fradique Coutinho | não crítica | 2026-08-04T04:00:00 |
| SIM-ECR-001 | Luz | não crítica | 2026-08-05T00:00:00 |
| SIM-ECR-005 | Oscar Freire | não crítica | 2026-08-05T10:00:00 |
| SIM-ECR-004 | Paulista | não crítica | 2026-08-06T20:00:00 |
| SIM-ECR-019 | Higienópolis-Mackenzie | não crítica | 2026-08-08T05:00:00 |
| SIM-ECR-030 | Fradique Coutinho | crítica | 2026-08-08T10:00:00 |


### Turno de 8 horas

**Agenda**

| id_escada | estacao | grupo | primeiro_alerta | rota_ida | rota_volta | inicio_minutos | fim_minutos | minutos_servico |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| SIM-ECR-027 | Higienópolis-Mackenzie | crítica | 2026-08-04T05:00:00 | Luz -> República -> Higienópolis-Mackenzie | Higienópolis-Mackenzie -> República -> Luz | 0 | 72 | 60 |
| SIM-ECR-024 | Pinheiros | crítica | 2026-08-04T18:00:00 | Luz -> República -> Higienópolis-Mackenzie -> Paulista -> Oscar Freire -> Fradique Coutinho -> Faria Lima -> Pinheiros | Pinheiros -> Faria Lima -> Fradique Coutinho -> Oscar Freire -> Paulista -> Higienópolis-Mackenzie -> República -> Luz | 72 | 170 | 60 |
| SIM-ECR-012 | Paulista | crítica | 2026-08-06T21:00:00 | Luz -> República -> Higienópolis-Mackenzie -> Paulista | Paulista -> Higienópolis-Mackenzie -> República -> Luz | 170 | 246 | 60 |
| SIM-ECR-030 | Fradique Coutinho | crítica | 2026-08-08T10:00:00 | Luz -> República -> Higienópolis-Mackenzie -> Paulista -> Oscar Freire -> Fradique Coutinho | Fradique Coutinho -> Oscar Freire -> Paulista -> Higienópolis-Mackenzie -> República -> Luz | 246 | 334 | 60 |
| SIM-ECR-016 | Pinheiros | não crítica | 2026-08-01T03:00:00 | Luz -> República -> Higienópolis-Mackenzie -> Paulista -> Oscar Freire -> Fradique Coutinho -> Faria Lima -> Pinheiros | Pinheiros -> Faria Lima -> Fradique Coutinho -> Oscar Freire -> Paulista -> Higienópolis-Mackenzie -> República -> Luz | 334 | 412 | 40 |
| SIM-ECR-022 | Fradique Coutinho | não crítica | 2026-08-02T16:00:00 | Luz -> República -> Higienópolis-Mackenzie -> Paulista -> Oscar Freire -> Fradique Coutinho | Fradique Coutinho -> Oscar Freire -> Paulista -> Higienópolis-Mackenzie -> República -> Luz | 412 | 480 | 40 |

**Pendentes**

| id_escada | estacao | grupo | primeiro_alerta |
| --- | --- | --- | --- |
| SIM-ECR-013 | Oscar Freire | não crítica | 2026-08-03T21:00:00 |
| SIM-ECR-014 | Fradique Coutinho | não crítica | 2026-08-04T04:00:00 |
| SIM-ECR-001 | Luz | não crítica | 2026-08-05T00:00:00 |
| SIM-ECR-005 | Oscar Freire | não crítica | 2026-08-05T10:00:00 |
| SIM-ECR-004 | Paulista | não crítica | 2026-08-06T20:00:00 |
| SIM-ECR-019 | Higienópolis-Mackenzie | não crítica | 2026-08-08T05:00:00 |

### Redução de 8 para 7 horas

**Agenda após a redução**

| id_escada | estacao | grupo | primeiro_alerta | rota_ida | rota_volta | inicio_minutos | fim_minutos | minutos_servico |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| SIM-ECR-027 | Higienópolis-Mackenzie | crítica | 2026-08-04T05:00:00 | Luz -> República -> Higienópolis-Mackenzie | Higienópolis-Mackenzie -> República -> Luz | 0 | 72 | 60 |
| SIM-ECR-024 | Pinheiros | crítica | 2026-08-04T18:00:00 | Luz -> República -> Higienópolis-Mackenzie -> Paulista -> Oscar Freire -> Fradique Coutinho -> Faria Lima -> Pinheiros | Pinheiros -> Faria Lima -> Fradique Coutinho -> Oscar Freire -> Paulista -> Higienópolis-Mackenzie -> República -> Luz | 72 | 170 | 60 |
| SIM-ECR-012 | Paulista | crítica | 2026-08-06T21:00:00 | Luz -> República -> Higienópolis-Mackenzie -> Paulista | Paulista -> Higienópolis-Mackenzie -> República -> Luz | 170 | 246 | 60 |
| SIM-ECR-030 | Fradique Coutinho | crítica | 2026-08-08T10:00:00 | Luz -> República -> Higienópolis-Mackenzie -> Paulista -> Oscar Freire -> Fradique Coutinho | Fradique Coutinho -> Oscar Freire -> Paulista -> Higienópolis-Mackenzie -> República -> Luz | 246 | 334 | 60 |
| SIM-ECR-016 | Pinheiros | não crítica | 2026-08-01T03:00:00 | Luz -> República -> Higienópolis-Mackenzie -> Paulista -> Oscar Freire -> Fradique Coutinho -> Faria Lima -> Pinheiros | Pinheiros -> Faria Lima -> Fradique Coutinho -> Oscar Freire -> Paulista -> Higienópolis-Mackenzie -> República -> Luz | 334 | 412 | 40 |

**Pendentes após a redução**

| id_escada | estacao | grupo | primeiro_alerta |
| --- | --- | --- | --- |
| SIM-ECR-013 | Oscar Freire | não crítica | 2026-08-03T21:00:00 |
| SIM-ECR-014 | Fradique Coutinho | não crítica | 2026-08-04T04:00:00 |
| SIM-ECR-001 | Luz | não crítica | 2026-08-05T00:00:00 |
| SIM-ECR-005 | Oscar Freire | não crítica | 2026-08-05T10:00:00 |
| SIM-ECR-004 | Paulista | não crítica | 2026-08-06T20:00:00 |
| SIM-ECR-019 | Higienópolis-Mackenzie | não crítica | 2026-08-08T05:00:00 |
| SIM-ECR-022 | Fradique Coutinho | não crítica | 2026-08-02T16:00:00 |


### Interpretação dos planos

| Turno | Limite em minutos | Usados | Atendidos | Pendentes |
| --- | --- | --- | --- | --- |
| 4 horas | 240 | 238 | 3 | 9 |
| Reduzido para 3 horas | 180 | 170 | 2 | 10 |
| 6 horas | 360 | 334 | 4 | 8 |
| Reduzido para 5 horas | 300 | 246 | 3 | 9 |
| 8 horas | 480 | 480 | 6 | 6 |
| Reduzido para 7 horas | 420 | 412 | 5 | 7 |

Os 12 equipamentos em alerta ficam distribuídos entre agenda e pendentes. Todos os planos respeitam a duração disponível. A redução retira um atendimento em cada cenário. O turno de oito horas ocupa toda a capacidade; nos demais cenários sobra tempo insuficiente para um atendimento adicional entre os pendentes.

Este procedimento segue as regras de fila e não promete maximizar a quantidade atendida. Retornar à base após cada serviço e priorizar os grupos são premissas da atividade.

## Conclusão

A solução demonstra geração reproduzível, buscas manuais e uso de filas e pilhas com listas. A busca binária consulta menos posições nos experimentos apresentados, desde que a lista esteja ordenada. O plano respeita o tempo disponível e transfere para pendentes os atendimentos que não cabem.

As conclusões se referem a dados sintéticos e deslocamentos simulados. Os perfis auxiliares são usados para auditoria, sem substituir a detecção de alertas pelos valores dos sensores. Os CSVs e os módulos Python acompanham o notebook no repositório.